# Armenia modular runner

Run this notebook from the project root.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

from src.armenia_modular.config import ASSETS_DIR, CITY_CENTERS_MAP_ASSET_NAME, CITY_COMPARISON_HTML_NAME, HERO_IMAGE_ASSET_NAME, INTERACTIVE_DIR, PROJECT_PDF_ASSET_NAME, PROJECT_POSTER_ASSET_NAME, THEORETICAL_DASHBOARD_HTML_NAME
from src.armenia_modular.pipeline import run_pipeline
from src.armenia_modular.interactive_fast import write_yerevan_single_polygon_html
from src.armenia_modular.interactive_precomputed import write_yerevan_precomputed_html
from src.armenia_modular.compare_business_areas import main as build_compare_business_areas
from src.armenia_modular.dashboard_embed import write_dashboard_html
from src.armenia_modular.site_builder import write_full_scrolly_site

In [ ]:
bundle = run_pipeline(save_master_csv=True)

master = bundle["master"]
logit_model = bundle["logit_model"]
features = bundle["features"]
means = bundle["means"]
stds = bundle["stds"]
mu0x = bundle["mu0x"]
mu0y = bundle["mu0y"]

master.head()

In [ ]:
single_html = write_yerevan_single_polygon_html(
    master=master,
    logit_model=logit_model,
    features=features,
    means=means,
    stds=stds,
    mu0x=mu0x,
    mu0y=mu0y,
)
single_html

In [ ]:
precomputed_html = write_yerevan_precomputed_html(
    master=master,
    logit_model=logit_model,
    features=features,
    means=means,
    stds=stds,
    mu0x=mu0x,
    mu0y=mu0y,
)
precomputed_html

In [ ]:
dashboard_html = write_dashboard_html(INTERACTIVE_DIR / THEORETICAL_DASHBOARD_HTML_NAME)
compare_html = build_compare_business_areas(
    yerevan_precomp_html=precomputed_html,
    out_html=INTERACTIVE_DIR / CITY_COMPARISON_HTML_NAME,
)

html_filled = Path(single_html).read_text(encoding="utf-8")
site_paths = write_full_scrolly_site(
    out_dir=str(INTERACTIVE_DIR),
    interactive_html=html_filled,
    viz_filename=Path(single_html).name,
    landing_filename="index.html",
    title="Yerevan scrolly",
    dashboard_filename=Path(dashboard_html).name,
    compare_html_src_path=str(compare_html),
    compare_filename=Path(compare_html).name,
    hero_image_src_path=str(ASSETS_DIR / HERO_IMAGE_ASSET_NAME),
    poster_svg_src_path=str(ASSETS_DIR / PROJECT_POSTER_ASSET_NAME),
    poster_title="",
    poster_sub=" ",
    pdf_src_path=str(ASSETS_DIR / PROJECT_PDF_ASSET_NAME),
    pdf_dst_name=PROJECT_PDF_ASSET_NAME,
    pdf_button_label="Open PDF",
    context_image_src_path=str(ASSETS_DIR / CITY_CENTERS_MAP_ASSET_NAME),
)

{
    "single_html": single_html,
    "precomputed_html": precomputed_html,
    "dashboard_html": str(dashboard_html),
    "compare_html": str(compare_html),
    "site_paths": site_paths,
}